<a href="https://colab.research.google.com/github/google-deepmind/tapnet/blob/main/tapnet/tapvidmv/droid/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Multi-View 3D Tracking Pipeline

This notebook is a **thin orchestration layer** — all algorithm code lives in the GitHub repo.

> **Benchmark Evaluation Rule:** Any model trained or fine-tuned on **any** DROID data must **not** report zero-shot results on the **TAPVid-MV (DROID)** evaluation split.

**Three stages** controlled by global flags (each can compute locally or load pre-computed outputs from `GCS_OUTPUT`):

| Stage | Compute | Load from GCS | Output |
|---|---|---|---|
| 1. Depth | `compute_depth.py` | `$GCS_OUTPUT/depth/` | Stereo depth + gripper refinement |
| 2. Extrinsics | `compute_extrinsics.py` | `$GCS_OUTPUT/extrinsics/` | Camera-robot alignment + quality metrics |
| 3. Tracks | `compute_tracks.py` | `$GCS_OUTPUT/tracks/` | Static BG + Robot 3D tracks |

---
## 0. Environment Setup

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
  if not os.path.exists("/content/tapnet"):
    subprocess.run(
      ["git", "clone", "--depth", "1", "https://github.com/google-deepmind/tapnet.git", "/content/tapnet"],
      check=True,
    )
  os.chdir("/content/tapnet/tapnet/tapvidmv/droid")

REPO_DIR = os.getcwd()
CACHE_DIR = os.path.join(REPO_DIR, "data", "cache")
GCS_OUTPUT = ""  # Optional: gs://... URI holding pre-computed Stage 1-3 outputs
print(f"cache: {CACHE_DIR}")

In [ ]:
COMPUTE_DEPTH = True
COMPUTE_EXTRINSICS = True
COMPUTE_TRACKS = True

print(f"COMPUTE_DEPTH      = {COMPUTE_DEPTH}")
print(f"COMPUTE_EXTRINSICS = {COMPUTE_EXTRINSICS}")
print(f"COMPUTE_TRACKS     = {COMPUTE_TRACKS}")

In [ ]:
if IN_COLAB:
  from google.colab import auth

  auth.authenticate_user()

  subprocess.run([sys.executable, "-m", "pip", "install", "-U", "ipython"], check=True)

  subprocess.run(["bash", "setup.sh"] + ([] if COMPUTE_DEPTH else ["--no-depth"]), check=True)
else:
  print("[SKIP] local checkout — dependencies come from setup.sh / the venv")

In [ ]:
import random

import matplotlib.pyplot as plt
import mediapy as media
import numpy as np
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
%reload_ext autoreload
%autoreload 2

from config import get_config
import core.depth
import core.geometry
import core.io
import core.physics
import core.runner
import core.tracks
import core.visualization
import compute_depth
import compute_extrinsics
import compute_tracks

config = get_config()

In [ ]:
serials_db, id_to_path, extrinsics_db, all_valid_ids = core.io.load_metadata(config)

valid_ids = sorted(core.runner.finished(config.paths.tracks, "tracks.npz"))
if not valid_ids and GCS_OUTPUT:
  out = subprocess.run(["gcloud", "storage", "ls", f"{GCS_OUTPUT}/tracks/"], check=True, capture_output=True, text=True)
  valid_ids = sorted(p.rstrip("/").rsplit("/", 1)[-1] for p in out.stdout.splitlines() if p.endswith("/"))
if not valid_ids and COMPUTE_DEPTH and COMPUTE_EXTRINSICS and COMPUTE_TRACKS:
  valid_ids = all_valid_ids
if not valid_ids:
  raise ValueError("No local tracks found; set GCS_OUTPUT = 'gs://...' or set COMPUTE_DEPTH/EXTRINSICS/TRACKS = True.")
print(f"{len(valid_ids)} candidate episodes available")

In [ ]:
episode_id = random.choice(valid_ids)

print(f"Episode: {episode_id}")

In [ ]:
episode = compute_depth.init_episode(episode_id, config.paths.raw, id_to_path, serials_db)

print(f"episode loaded: {list(episode['camera'].keys())}")

---
## 1. Stage 1: Depth

Stereo depth via S2M2 + SAM gripper refinement.

In [ ]:
if COMPUTE_DEPTH:
  s2m2_model, sam_predictor = compute_depth.init_all_models(device)

  episode = compute_depth.extract_svo_video(episode, config.depth.min_frames, config.depth.max_frames)
  episode = compute_depth.parse_robot_kinematics(episode)
  episode = compute_depth.align_temporal_streams(episode)
  episode = core.depth.compute_stereo_depth(episode, s2m2_model, device, config.depth.conf_thresh)
  episode = core.depth.refine_gripper_depth(episode, sam_predictor, config.depth)

  print("Stage 1 (Depth) COMPUTE complete")
else:
  if not GCS_OUTPUT:
    raise ValueError("Set GCS_OUTPUT = 'gs://...' or set COMPUTE_DEPTH = True.")
  depth_root = os.path.join(CACHE_DIR, "depth")
  src = f"{GCS_OUTPUT}/depth/{episode_id}"
  dst = os.path.join(depth_root, episode_id)
  subprocess.run(["gcloud", "storage", "rsync", "-r", src, dst], check=True)

  episode = core.io.load_depth_data(episode_id, depth_root, load_video=True)

  print("Stage 1 LOADED from GCS")

In [ ]:
core.visualization.inspect_dict_structure(episode)

frames = core.visualization.render_multicam_disparity_video(episode, max_frames=30)
media.show_video(frames, fps=10, title="Depth [left | right | disparity] per camera")

In [ ]:
core.visualization.show_gripper_refinement(episode, t=0)

cam_data = episode["camera"][episode["meta"]["wrist_cam_id"]]
core.visualization.show_distilled_gripper_3d(
  median_depth=cam_data["empirical_gripper_depth"],
  K=cam_data["K"],
  img_rgb=cam_data["video_rgb"][0],
)

---
## 2. Stage 2: Extrinsics

Dataset extrinsics → joint pixel-space alignment of all cameras.

In [ ]:
pb_renderer = core.physics.PyBulletRenderer(config.paths.urdf, gpu=config.render.gpu)
print(f"PyBulletRenderer EGL: {pb_renderer.gpu}")
render_pool = core.physics.RenderPool(pb_renderer, config.render.workers)

if COMPUTE_EXTRINSICS:
  poses = compute_extrinsics.init_camera_states(episode, extrinsics_db)
  poses = compute_extrinsics.joint_alignment(episode, poses, render_pool, device, config)

  print("Stage 2 (Extrinsics) COMPUTE complete")
else:
  if not GCS_OUTPUT:
    raise ValueError("Set GCS_OUTPUT = 'gs://...' or set COMPUTE_EXTRINSICS = True.")
  ext_root = os.path.join(CACHE_DIR, "extrinsics")
  src = f"{GCS_OUTPUT}/extrinsics/{episode_id}"
  dst = os.path.join(ext_root, episode_id)
  subprocess.run(["gcloud", "storage", "rsync", "-r", src, dst], check=True)

  poses = core.io.load_extrinsics(episode, ext_root)

  print("Stage 2 LOADED from GCS")

In [ ]:
if COMPUTE_EXTRINSICS:
  metrics = compute_extrinsics.episode_metrics(episode, poses, render_pool, device, config)
else:
  metrics = core.io.load_metrics(episode_id, ext_root)

print(f"Episode: {metrics.pop('episode_id')}\n{'=' * 60}")
for k, v in metrics.items():
  val = f"{v:.4f}" if isinstance(v, float) else v
  print(f"  {k:45s} = {val}")

In [ ]:
axes_frames = core.visualization.render_cross_camera_axes(episode, poses, max_frames=30)
media.show_video(axes_frames, fps=10, title="Camera Axes Overlay")

In [ ]:
seg_frames = core.visualization.render_segmentation_video(episode, poses, pb_renderer, max_frames=30)
media.show_video(seg_frames, fps=10, title="Robot Mask Overlay")

In [ ]:
core.visualization.show_fused_point_cloud(
  episode, poses, t=0, max_depth=config.extrinsics.max_depth, height=600, width=1000
)

In [ ]:
orbit_frames = core.visualization.render_4d_orbit_with_tracks(
  episode, poses, max_depth=config.extrinsics.max_depth, max_frames=30
)
media.show_video(orbit_frames, fps=10, title="4D Orbit")

---
## 3. Stage 3: Tracking

**Static Background + Robot Tracks** — no tracker model dependency.

**Dual-Track Architecture:**
- **Track A (Static Background)**: Multi-view depth consensus → fixed world 3D → project to 2D per-view using extrinsics (static prior)
- **Track B (Robot)**: URDF forward kinematics → per-link binding → cross-view projection

In [ ]:
if COMPUTE_TRACKS:
  compute_tracks.check_robot_loss(metrics, config.tracks.max_robot_loss)
  queries = compute_tracks.query_frames(episode, poses, pb_renderer, config)
  candidates = compute_tracks.find_candidates(episode, poses, pb_renderer, queries, config)
  tracks = compute_tracks.label_tracks(candidates, episode, poses, pb_renderer, config)

  print("Stage 3 (Tracks) COMPUTE complete")
else:
  if not GCS_OUTPUT:
    raise ValueError("Set GCS_OUTPUT = 'gs://...' or set COMPUTE_TRACKS = True.")
  tracks_root = os.path.join(CACHE_DIR, "tracks")
  src = f"{GCS_OUTPUT}/tracks/{episode_id}"
  dst = os.path.join(tracks_root, episode_id)
  subprocess.run(["gcloud", "storage", "rsync", "-r", src, dst], check=True)

  tracks = core.io.load_track_data(episode_id, tracks_root)

  print("Stage 3 LOADED from GCS")

tracks = core.tracks.sample_tracks(tracks, config.tracks.points_per_class)
uv = core.tracks.project_views(tracks["tracks_3d"], episode, poses)
tracks_3d, vis = tracks["tracks_3d"], tracks["vis"]
query_view, n_robot, n_static = tracks["query_view"], tracks["n_robot"], tracks["n_static"]

In [ ]:
cam_ids = list(episode["camera"])

ref_view = 1
y_robot = uv[ref_view][0, :n_robot, 1]
norm_r = plt.Normalize(y_robot.min(), y_robot.max())
robot_colors = (plt.cm.gist_rainbow(norm_r(y_robot))[:, :3] * 255).astype(np.uint8)

y_static = uv[ref_view][0, n_robot:, 1]
norm_s = plt.Normalize(y_static.min(), y_static.max())
static_colors = (plt.cm.gist_rainbow(norm_s(y_static))[:, :3] * 255).astype(np.uint8)

combined_colors = np.concatenate([robot_colors, static_colors], axis=0)

all_frames = []
for view, cam_id in enumerate(cam_ids):
  frames = core.visualization.render_2d_tracking_video(
    episode["camera"][cam_id]["video_rgb"],
    uv[view],
    vis[view],
    global_colors=combined_colors,
    tgt_size=(256, 456),
    linewidth=1,
  )
  for f in frames:
    core.visualization.draw_label(f, f"Cam [{cam_id[:8]}]", (10, 25), 0.6, (255, 255, 255), 1, 3)
  all_frames.append(np.array(frames))

combined = np.concatenate(all_frames, axis=2)
media.show_video(
  combined,
  fps=10,
  title=f"All Tracks ({n_robot} robot + {n_static} static, rainbow by image height) — All Cameras",
)

In [ ]:
print(f"Using 'tracks_3d', shape={tracks_3d.shape}")

orbit_frames = core.visualization.render_4d_orbit_with_tracks(
  episode, poses, tracks_3d=tracks_3d, max_depth=config.extrinsics.max_depth
)
media.show_video(orbit_frames, fps=10, title="4D Orbit — Point Cloud + Tracks + Cameras")